# Intrinsic Wavefront Analysis and Plots

**Author:** Aaron Roodman  
**Date Created:** 2026-02-23  
**Last Modified:** 2026-03-17  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Full Array Mode, Zernike, Analysis

## Description

This notebook analyzes the FAM Zernike table created by `intrinsics_mktable.ipynb`.

**Analysis includes:**
1. Load parquet file with Zernike measurements and visit_info companion table
2. Identify FAM blocks (contiguous triplets with visits separated by 3) and print summary
3. Filter out sparse days (< 5 FAM triplet seq_num)
4. Robust per-image linear fit (constant + x,y slopes) to data-model residuals using RLM
5. Plot fit parameters and residual histograms per day_obs
6. Single-image residual maps and histograms for Z4-Z11
7. Create hexbin visualizations: Data, Model, and Data-Model residuals

**Input:** Parquet files created by mktable notebook (in `output/`)  
**Output:** Trio comparison plots (PNG), fit parameter plots, single-image residual maps, FAM block summary

**Based on:** `intrinsics_mktable.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-23 | Aaron Roodman | Initial version |
| 2026-03-15 | Aaron Roodman | Added CCS/OCS coord_sys parameter; per-image linear fit; fit parameter plots; rotator angle filter |
| 2026-03-17 | Aaron Roodman | Switched to robust RLM; FAM block analysis; single-image residual maps; MPEG movie; multi-page PDFs |
| 2026-03-19 | Aaron Roodman | Generalized focal-plane Zernike fit method with Noll convention basis (normalized to fp_radius=1.75 deg); two fits per image: z13 (Noll 1-3: piston+tilt+tip) and z16 (Noll 1-6: adds defocus+astigmatism); fit output includes coefficients, bse errors, RLM scale per Zernike; merged output table (intrinsic_fits_ parquet) with visit_info; trio plots use median and go into single PDF; clean up individual JPEGs after successful movie creation |
| 2026-03-19 | Aaron Roodman | Read Noll indices from visit_info nollIndices column instead of guessing; all plots and fit column names use actual Noll indices from data; added z16 fit parameter plots alongside z13; plot Zernike sets derived from data (iZs_plot_12) instead of hardcoded [4..15] |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load Data](#load)
4. [FAM Block Analysis](#blocks)
5. [Helper Functions](#functions)
6. [Analysis](#analysis)
7. [Focal-Plane Zernike Fits](#fit)
8. [Output Fit Table](#fittable)
9. [Fit Parameter Plots & Residual Histograms](#fitplots)
10. [Single-Image Residual Maps](#singleimage)
11. [Trio Comparison Plots](#viz)

<a id='params'></a>
## Parameters

In [ ]:
from pathlib import Path

# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system: 'CCS' or 'OCS' (must match mktable choice)
coord_sys = 'OCS'

# Parquet file version strings (must match mktable output)
wep_ver = 'wep_v16_8_0'
dviz_ver = 'dviz_v3_5_0'

# Parquet file dates
day_obs_min = 20251020
day_obs_max = 20251231

# Date range string for plot titles
date_range_str = f'{day_obs_min}-{day_obs_max}'

# Rotator angle filter (degrees) — only keep images within this range
rotator_angle_min = -3.0
rotator_angle_max = 3.0

# Minimum unique seq_num per day_obs to keep (filters sparse days)
min_seq_num_per_day = 5

# Zernike indices: inferred from data after loading (see Load Data section)
# iZs and iZidx are set dynamically below

# Plotting parameters
plo_default = 4.0   # Low percentile for colorbar
phi_default = 96.0  # High percentile for colorbar
output_dir = 'output'

# Ensure output directory exists
Path(output_dir).mkdir(parents=True, exist_ok=True)

<a id='setup'></a>
## Setup & Imports

In [ ]:
# Standard imports
from matplotlib import pyplot as plt
from matplotlib import lines
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
import pandas as pd
from scipy.stats import binned_statistic_2d, binned_statistic
from pathlib import Path
import statsmodels.api as sm
import subprocess

# Astropy
from astropy.table import Table, QTable

# Set plot defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Pandas display options
pd.options.display.max_columns = None
pd.options.display.width = None

<a id='load'></a>
## Load Data

In [ ]:
# Load the main donut parquet file and visit_info companion table
parquet_file = f'{output_dir}/fam_zernikes_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}.parquet'
visit_info_file = f'{output_dir}/fam_zernikes_{wep_ver}_{dviz_ver}_{day_obs_min}_{day_obs_max}_visits.parquet'

if not Path(parquet_file).exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_file}\n"
                           f"Run intrinsics_mktable.ipynb first to create it.")

print(f"Loading data from: {parquet_file}")
aosTable = QTable.read(parquet_file)
print(f"Loaded {len(aosTable)} rows, {len(aosTable.columns)} columns")
print(f"Coordinate system: {coord_sys}")

# Load visit_info table
if Path(visit_info_file).exists():
    visit_info = QTable.read(visit_info_file)
    print(f"\nLoaded visit_info from: {visit_info_file}")
    print(f"  {len(visit_info)} visits, columns: {visit_info.colnames}")
else:
    visit_info = None
    print(f"\nWarning: visit_info file not found: {visit_info_file}")
    print("  FAM block analysis will be skipped.")

In [3]:
# Display table summary
print("\nTable columns:")
print(sorted(aosTable.columns))

# Count by day_obs
print("\nMeasurements per day_obs:")
day_counts = pd.DataFrame({'day_obs': aosTable['day_obs']}).value_counts().sort_index()
print(day_counts)


Table columns:
['centroid_x', 'centroid_x_extra', 'centroid_x_intra', 'centroid_y', 'centroid_y_extra', 'centroid_y_intra', 'coord_dec', 'coord_dec_extra', 'coord_dec_intra', 'coord_ra', 'coord_ra_extra', 'coord_ra_intra', 'day_obs', 'detector', 'extra_fpx', 'extra_fpy', 'intra_fpx', 'intra_fpy', 'matched_intra_extra', 'seq_num', 'th_N', 'th_N_extra', 'th_N_intra', 'th_W', 'th_W_extra', 'th_W_intra', 'thx_CCS', 'thx_CCS_extra', 'thx_CCS_intra', 'thx_OCS', 'thx_OCS_extra', 'thx_OCS_intra', 'thy_CCS', 'thy_CCS_extra', 'thy_CCS_intra', 'thy_OCS', 'thy_OCS_extra', 'thy_OCS_intra', 'used', 'zk_CCS', 'zk_CCS_mean', 'zk_NW', 'zk_OCS', 'zk_intrinsic', 'zk_residual']

Measurements per day_obs:
day_obs 
20250825    120458
20250826     94567
20250907      1586
20250909      7029
20250912     11256
20250913     52244
20251023     44375
20251024      9162
20251026     79514
20251027     20498
20251028      7374
Name: count, dtype: int64


In [4]:
# Display sample rows with formatted floats
# Select scalar columns only
scalar_cols = []
for col in aosTable.columns:
    if not hasattr(aosTable[col][0], '__len__') or isinstance(aosTable[col][0], str):
        scalar_cols.append(col)

df_display = aosTable[scalar_cols].to_pandas()
pd.options.display.float_format = '{:.2f}'.format

print("\nSample rows (first 5):")
print(df_display.head())

pd.reset_option('display.float_format')


Sample rows (first 5):
  detector  used  coord_ra  coord_dec  centroid_x  centroid_y  thx_CCS  \
0  R01_S00  True      4.72      -0.38     3773.04     3752.49    -0.03   
1  R01_S01  True      4.72      -0.38     1991.94     3338.47    -0.03   
2  R01_S01  True      4.72      -0.38     2997.63     2987.60    -0.03   
3  R01_S01  True      4.72      -0.38     2273.00     2673.47    -0.03   
4  R01_S01  True      4.72      -0.38      988.99     2905.48    -0.03   

   thy_CCS  thx_OCS  thy_OCS  th_N  th_W  coord_ra_intra  coord_ra_extra  \
0    -0.01    -0.03    -0.01  0.03 -0.02            4.72            4.72   
1    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
2    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
3    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   
4    -0.01    -0.03    -0.01  0.02 -0.02            4.72            4.72   

   coord_dec_intra  coord_dec_extra  centroid_x_intra  centroid_x_extra  \

<a id='blocks'></a>
## FAM Block Analysis

Identify contiguous blocks of Full Array Mode (FAM) triplets from the visit_info table.
FAM triplets (intra, in-focus, extra-focal) have visits separated by increments of 3.

In [ ]:
# Identify FAM blocks from visit_info
# FAM triplets (intra, in-focus, extra) have visits separated by increments of 3.
# A FAM block is a contiguous group of visits where consecutive visits differ by exactly 3.

if visit_info is not None:
    visits_sorted = visit_info.copy()
    visits_sorted.sort('visit')
    visit_arr = np.array(visits_sorted['visit'])
    
    # Get mean Z4 (first element of zk_OCS) per visit from the main table
    day_obs_main = np.array(aosTable['day_obs'])
    seq_num_main = np.array(aosTable['seq_num'])
    zk_data_all = np.stack(aosTable[f'zk_{coord_sys}'])
    
    # Compute mean Z4 per visit
    visit_z4_mean = {}
    for row in visits_sorted:
        dobs = row['day_obs']
        snum = row['seq_num']
        mask = (day_obs_main == dobs) & (seq_num_main == snum)
        if np.sum(mask) > 0:
            visit_z4_mean[(dobs, snum)] = np.mean(zk_data_all[mask, 0])
        else:
            visit_z4_mean[(dobs, snum)] = np.nan
    
    # Find block boundaries: consecutive visits differ by 3
    diffs = np.diff(visit_arr)
    block_breaks = np.where(diffs != 3)[0] + 1
    block_starts = np.concatenate([[0], block_breaks])
    block_ends = np.concatenate([block_breaks, [len(visit_arr)]])
    
    print(f"Found {len(block_starts)} FAM blocks from {len(visit_arr)} visits\n")
    
    # Build summary table
    block_summary = []
    for b_idx, (bs, be) in enumerate(zip(block_starts, block_ends)):
        block_rows = visits_sorted[bs:be]
        n_visits = len(block_rows)
        dobs = int(block_rows['day_obs'][0])
        seq_min = int(np.min(block_rows['seq_num']))
        seq_max = int(np.max(block_rows['seq_num']))
        band = str(block_rows['band'][0])
        alt_mean = float(np.mean(block_rows['alt']))
        az_mean = float(np.mean(block_rows['az']))
        rot_mean = float(np.mean(block_rows['rotAngle']))
        
        # Z4 of first and last image in block
        first_key = (int(block_rows['day_obs'][0]), int(block_rows['seq_num'][0]))
        last_key = (int(block_rows['day_obs'][-1]), int(block_rows['seq_num'][-1]))
        z4_first = visit_z4_mean.get(first_key, np.nan)
        z4_last = visit_z4_mean.get(last_key, np.nan)
        
        block_summary.append({
            'block': b_idx,
            'day_obs': dobs,
            'seq_min': seq_min,
            'seq_max': seq_max,
            'n_visits': n_visits,
            'band': band,
            'alt': alt_mean,
            'az': az_mean,
            'rotAngle': rot_mean,
            'Z4_first': z4_first,
            'Z4_last': z4_last,
        })
    
    block_df = pd.DataFrame(block_summary)
    pd.options.display.float_format = '{:.2f}'.format
    print("FAM Block Summary:")
    print(block_df.to_string(index=False))
    pd.reset_option('display.float_format')
else:
    print("visit_info not available — skipping FAM block analysis")

In [ ]:
# Filter out day_obs with fewer than min_seq_num_per_day unique seq_num
# These are days with too few FAM triplets for reliable analysis
day_obs_all = np.array(aosTable['day_obs'])
seq_num_all = np.array(aosTable['seq_num'])

unique_days = sorted(set(day_obs_all.tolist()))
print(f"Before filtering: {len(unique_days)} days, {len(aosTable)} donuts")
print(f"Unique seq_num per day_obs (minimum required: {min_seq_num_per_day}):")

sparse_days = []
for d in unique_days:
    n_unique_seq = len(set(seq_num_all[day_obs_all == d].tolist()))
    n_donuts_day = np.sum(day_obs_all == d)
    status = "REMOVED" if n_unique_seq < min_seq_num_per_day else "kept"
    print(f"  {d}: {n_unique_seq} seq_num, {n_donuts_day} donuts  [{status}]")
    if n_unique_seq < min_seq_num_per_day:
        sparse_days.append(d)

if sparse_days:
    sparse_mask = ~np.isin(day_obs_all, sparse_days)
    n_before = len(aosTable)
    aosTable = aosTable[sparse_mask]
    n_after = len(aosTable)
    print(f"\nRemoved {len(sparse_days)} day_obs with < {min_seq_num_per_day} unique seq_num: {sparse_days}")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
else:
    print(f"\nAll day_obs have >= {min_seq_num_per_day} unique seq_num")

# Summary of remaining data
remaining_days = sorted(set(np.array(aosTable['day_obs']).tolist()))
n_images = len(set(zip(np.array(aosTable['day_obs']).tolist(), np.array(aosTable['seq_num']).tolist())))
print(f"\nRemaining: {len(remaining_days)} days, {n_images} images, {len(aosTable)} donuts")

In [ ]:
# Filter by rotator angle
if 'rotator_angle' in aosTable.columns:
    rot_angles = np.array(aosTable['rotator_angle'])
    rot_mask = (rot_angles >= rotator_angle_min) & (rot_angles <= rotator_angle_max)
    n_before = len(aosTable)
    aosTable = aosTable[rot_mask]
    n_after = len(aosTable)
    n_images_before = len(set(zip(np.array(aosTable['day_obs']), np.array(aosTable['seq_num']))))
    print(f"Rotator angle filter [{rotator_angle_min}, {rotator_angle_max}] deg:")
    print(f"  Donuts: {n_before} -> {n_after} ({n_before - n_after} removed)")
    print(f"  Remaining images: {n_images_before}")
else:
    print("Warning: 'rotator_angle' column not found in table.")
    print("  Rotator angle filter NOT applied.")
    print("  Re-run intrinsics_mktable.ipynb to include rotator angles.")

<a id='functions'></a>
## Helper Functions

In [ ]:
def get_zernike(table, column_name, iZ):
    """Extract a single Zernike term from an array column.
    
    Parameters
    ----------
    table : QTable
        Table with Zernike array columns
    column_name : str
        Column name (e.g. 'zk_OCS', 'zk_intrinsic', 'zk_residual')
    iZ : int
        Zernike index (4-28, excluding 20, 21)
    
    Returns
    -------
    array : ndarray
        Zernike values
    """
    if iZ not in iZidx:
        raise ValueError(f"Zernike Z{iZ} not in table. Available: {iZs}")
    zk_array = np.stack(table[column_name])
    return zk_array[:, iZidx[iZ]]


def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Add a vertical color bar to an image plot."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1./aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


# ============================================================
# Focal-plane Zernike fitting
# ============================================================

def focal_plane_zernike_basis(thx_deg, thy_deg, max_noll, fp_radius=1.75):
    """Build focal-plane Noll Zernike basis matrix.
    
    Coordinates are normalized to the focal-plane radius so that the
    Zernike polynomials are evaluated on a unit disk.
    
    Parameters
    ----------
    thx_deg, thy_deg : ndarray
        Field angles in degrees.
    max_noll : int
        Maximum Noll index (1-6 supported).
    fp_radius : float
        Focal plane radius in degrees for normalization (default 1.75).
    
    Returns
    -------
    A : ndarray, shape (n_points, max_noll)
        Design matrix with one column per focal Zernike term.
    labels : list of str
        Labels for each column (e.g. 'Z1_piston', 'Z2_tilt', ...).
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    r2 = x**2 + y**2
    
    cols = []
    labels = []
    
    if max_noll >= 1:
        cols.append(np.ones_like(x))
        labels.append('Z1_piston')
    if max_noll >= 2:
        cols.append(2.0 * x)
        labels.append('Z2_tilt')
    if max_noll >= 3:
        cols.append(2.0 * y)
        labels.append('Z3_tip')
    if max_noll >= 4:
        cols.append(np.sqrt(3) * (2.0 * r2 - 1.0))
        labels.append('Z4_defocus')
    if max_noll >= 5:
        cols.append(2.0 * np.sqrt(6) * x * y)
        labels.append('Z5_astig45')
    if max_noll >= 6:
        cols.append(np.sqrt(6) * (x**2 - y**2))
        labels.append('Z6_astig0')
    
    return np.column_stack(cols), labels


def fit_focal_zernikes(aosTable_matched, iZs, iZidx, coord_sys,
                        max_focal_noll=3, include_intrinsic=True,
                        fp_radius=1.75, prefix='z13'):
    """Fit focal-plane Noll Zernikes to per-image data residuals using robust regression.
    
    For each image and each Zernike term iZ, fits:
        residual = c1*Zfocal_1 + c2*Zfocal_2 + ... + cN*Zfocal_N
    where residual = zk_data - zk_model*1e6 (if include_intrinsic) or just zk_data.
    
    Parameters
    ----------
    aosTable_matched : QTable
        Matched donut table.
    iZs : list of int
        Zernike indices to fit.
    iZidx : dict
        Mapping from iZ to column index in zk arrays.
    coord_sys : str
        Coordinate system ('OCS' or 'CCS').
    max_focal_noll : int
        Maximum focal Noll index (default 3 for piston+tilt+tip).
    include_intrinsic : bool
        If True, subtract intrinsic model before fitting (default True).
    fp_radius : float
        Focal plane radius in degrees (default 1.75).
    prefix : str
        Column name prefix for output (e.g. 'z13' or 'z16').
    
    Returns
    -------
    fit_table : QTable
        One row per image with columns:
        - day_obs, seq_num, image_idx, n_donuts
        - {prefix}_z{iZ}_c{j} : fit coefficient j for Zernike iZ
        - {prefix}_z{iZ}_c{j}_err : standard error of coefficient
        - {prefix}_z{iZ}_scale : RLM scale estimate (robust sigma)
    zk_fit_vals : ndarray, shape (n_donuts, n_zernikes)
        Per-donut fitted values.
    zk_rlm_weights : ndarray, shape (n_donuts, n_zernikes)
        Per-donut RLM weights.
    """
    thx_col = f'thx_{coord_sys}'
    thy_col = f'thy_{coord_sys}'
    
    day_obs_arr = np.array(aosTable_matched['day_obs'])
    seq_num_arr = np.array(aosTable_matched['seq_num'])
    thx_arr = np.rad2deg(np.array(aosTable_matched[thx_col]))
    thy_arr = np.rad2deg(np.array(aosTable_matched[thy_col]))
    zk_data = np.stack(aosTable_matched[f'zk_{coord_sys}'])
    zk_model = np.stack(aosTable_matched['zk_intrinsic'])
    
    images = sorted(set(zip(day_obs_arr.tolist(), seq_num_arr.tolist())))
    n_donuts = len(aosTable_matched)
    n_zernikes = len(iZs)
    n_coeffs = max_focal_noll
    
    zk_fit_vals = np.zeros((n_donuts, n_zernikes))
    zk_rlm_weights = np.ones((n_donuts, n_zernikes))
    fit_params_list = []
    
    for img_idx, (dobs, snum) in enumerate(images):
        mask = (day_obs_arr == dobs) & (seq_num_arr == snum)
        n_pts = np.sum(mask)
        
        # Build design matrix
        A, col_labels = focal_plane_zernike_basis(
            thx_arr[mask], thy_arr[mask], max_focal_noll, fp_radius)
        
        img_params = {'day_obs': dobs, 'seq_num': snum,
                      'image_idx': img_idx, 'n_donuts': int(n_pts)}
        
        for j_idx, iZ in enumerate(iZs):
            # Compute residual
            if include_intrinsic:
                resid = zk_data[mask, j_idx] - zk_model[mask, j_idx] * 1e6
            else:
                resid = zk_data[mask, j_idx].copy()
            
            # Robust fit
            try:
                rlm_model = sm.RLM(resid, A, M=sm.robust.norms.HuberT())
                rlm_results = rlm_model.fit()
                coeffs = rlm_results.params
                bse = rlm_results.bse
                scale = float(rlm_results.scale)
                weights = rlm_results.weights
            except Exception:
                coeffs, _, _, _ = np.linalg.lstsq(A, resid, rcond=None)
                bse = np.full(n_coeffs, np.nan)
                scale = float(np.std(resid - A @ coeffs))
                weights = np.ones(n_pts)
            
            # Store coefficients, errors, scale
            for ci in range(n_coeffs):
                img_params[f'{prefix}_z{iZ}_c{ci+1}'] = float(coeffs[ci])
                img_params[f'{prefix}_z{iZ}_c{ci+1}_err'] = float(bse[ci])
            img_params[f'{prefix}_z{iZ}_scale'] = scale
            
            # Per-donut fitted values
            zk_fit_vals[mask, j_idx] = A @ coeffs
            zk_rlm_weights[mask, j_idx] = weights
        
        fit_params_list.append(img_params)
    
    fit_table = QTable(fit_params_list)
    print(f"Fit '{prefix}' (focal Noll 1-{max_focal_noll}): "
          f"{len(images)} images, {n_donuts} donuts, "
          f"include_intrinsic={include_intrinsic}")
    
    return fit_table, zk_fit_vals, zk_rlm_weights


# ============================================================
# Plotting methods
# ============================================================

def plot_fit_params_and_residuals(fit_table_day, aosTable_matched, plot_mask_day,
                                  day_obs_list, fit_prefix='z13',
                                  iZs_fit_plot=None, iZs_hist=None,
                                  output_dir='.', show=True):
    """Plot fit coefficients vs image and residual histograms into a single PDF.
    
    Parameters
    ----------
    fit_table_day : QTable
        Fit parameters table filtered to selected days.
    aosTable_matched : QTable
        Full matched donut table (with zk_fit column).
    plot_mask_day : ndarray (bool)
        Mask into aosTable_matched for the selected days.
    day_obs_list : list of int
        Day_obs values being plotted (used in titles).
    fit_prefix : str
        Prefix for fit columns (e.g. 'z13').
    iZs_fit_plot : list of int, optional
        Zernike indices for fit param plots (default [4..15]).
    iZs_hist : list of int, optional
        Zernike indices for residual histograms (default [4..15]).
    output_dir : str
        Directory for saved figures.
    show : bool
        If True, display plots inline. If False, save and close.
    """
    if iZs_fit_plot is None:
        iZs_fit_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    if iZs_hist is None:
        iZs_hist = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    
    if len(day_obs_list) == 1:
        day_label = str(day_obs_list[0])
    elif len(day_obs_list) <= 4:
        day_label = ', '.join(str(d) for d in day_obs_list)
    else:
        day_label = f'{day_obs_list[0]}...{day_obs_list[-1]} ({len(day_obs_list)} days)'
    
    file_suffix = str(day_obs_list[0]) if len(day_obs_list) == 1 else 'all'
    
    # Determine how many coefficients this fit has
    first_iZ = iZs_fit_plot[0]
    sample_cols = [c for c in fit_table_day.colnames if c.startswith(f'{fit_prefix}_z{first_iZ}_c') and not c.endswith('_err')]
    n_coeffs = len(sample_cols)
    
    param_labels = ['c1 (piston)', 'c2 (tilt)', 'c3 (tip)',
                    'c4 (defocus)', 'c5 (astig45)', 'c6 (astig0)'][:n_coeffs]
    param_units = ['μm'] * n_coeffs
    
    pdf_path = f'{output_dir}/fit_params_resid_{fit_prefix}_{file_suffix}.pdf'
    
    with PdfPages(pdf_path) as pdf:
        image_idx = np.arange(len(fit_table_day))
        
        for ci in range(n_coeffs):
            cname = f'c{ci+1}'
            fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True)
            fig.suptitle(f'{fit_prefix} Fit: {param_labels[ci]} vs Image (day_obs: {day_label})', fontsize=14)
            
            for ax_idx, iZ in enumerate(iZs_fit_plot):
                ax = axes[ax_idx // 4, ax_idx % 4]
                col = f'{fit_prefix}_z{iZ}_{cname}'
                if col in fit_table_day.colnames:
                    vals = np.array(fit_table_day[col])
                    ax.plot(image_idx, vals, 'o-', markersize=3)
                ax.set_title(f'Z{iZ}')
                ax.set_ylabel(param_units[ci])
                ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
                if ax_idx // 4 == 2:
                    ax.set_xlabel('Image index')
            
            plt.tight_layout()
            pdf.savefig(fig, dpi=150, bbox_inches='tight')
            if show:
                plt.show()
            else:
                plt.close(fig)
        
        # Fit residual histograms
        zk_data_plot = np.stack(aosTable_matched[f'zk_{coord_sys}'])[plot_mask_day]
        zk_model_plot = np.stack(aosTable_matched['zk_intrinsic'])[plot_mask_day]
        fit_col = f'zk_fit_{fit_prefix}'
        if fit_col in aosTable_matched.colnames:
            zk_fit_plot = np.stack(aosTable_matched[fit_col])[plot_mask_day]
        else:
            zk_fit_plot = np.stack(aosTable_matched['zk_fit'])[plot_mask_day]
        
        fig, axes = plt.subplots(3, 4, figsize=(18, 10))
        fig.suptitle(f'{fit_prefix} Fit Residuals: data - model - fit (day_obs: {day_label})', fontsize=14)
        
        hist_range = (-1.0, 1.0)
        n_bins = 100
        
        for ax_idx, iZ in enumerate(iZs_hist):
            ax = axes[ax_idx // 4, ax_idx % 4]
            j = iZidx[iZ]
            resid = zk_data_plot[:, j] - zk_model_plot[:, j] * 1e6 - zk_fit_plot[:, j]
            n_total = len(resid)
            n_in = np.sum((resid >= hist_range[0]) & (resid <= hist_range[1]))
            n_out = n_total - n_in
            ax.hist(resid, bins=n_bins, range=hist_range, log=True,
                    edgecolor='black', linewidth=0.3, alpha=0.7)
            ax.set_title(f'Z{iZ}')
            ax.set_xlabel('μm')
            ax.set_ylabel('Count')
            ax.set_xlim(hist_range)
            std_val = np.std(resid)
            ax.text(0.97, 0.95, f'σ={std_val:.3f} μm\n{n_out}/{n_total} outside',
                    transform=ax.transAxes, ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        pdf.savefig(fig, dpi=150, bbox_inches='tight')
        if show:
            plt.show()
        else:
            plt.close(fig)
    
    print(f"Saved: {pdf_path}")


def plot_single_image_residual_grid(aosTable_matched, day_obs, seq_num,
                                     band='', fit_table=None, fit_prefix='z13',
                                     iZs_plot=None,
                                     n_steps=16, statistic='median',
                                     plo=2.0, phi=98.0,
                                     k1_range=1.0, fpradius=1.8,
                                     output_dir='.', show=True):
    """Plot a 3x4 grid of binned residual maps (Z4-Z15) for a single image.

    Uses GridSpec with fixed width ratios for consistent panel sizes.
    Includes k1 (c1 piston) bar indicator per subplot.
    """
    from matplotlib.gridspec import GridSpec
    from matplotlib.patches import Rectangle

    if iZs_plot is None:
        iZs_plot = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

    dobs_arr = np.array(aosTable_matched['day_obs'])
    snum_arr = np.array(aosTable_matched['seq_num'])
    img_mask = (dobs_arr == day_obs) & (snum_arr == seq_num)
    n_donuts = np.sum(img_mask)
    if n_donuts == 0:
        return None

    # Look up c1 (piston) values from fit_table
    c1_vals = {}
    if fit_table is not None:
        ft_dobs = np.array(fit_table['day_obs'])
        ft_snum = np.array(fit_table['seq_num'])
        ft_mask = (ft_dobs == day_obs) & (ft_snum == seq_num)
        if np.sum(ft_mask) == 1:
            ft_row = fit_table[ft_mask][0]
            for iZ in iZs_plot:
                col = f'{fit_prefix}_z{iZ}_c1'
                if col in fit_table.colnames:
                    c1_vals[iZ] = float(ft_row[col])

    xval = np.rad2deg(np.array(aosTable_matched[f'thy_{coord_sys}_extra'])[img_mask])
    yval = np.rad2deg(np.array(aosTable_matched[f'thx_{coord_sys}_extra'])[img_mask])
    zk_data_img = np.stack(aosTable_matched[f'zk_{coord_sys}'])[img_mask]
    zk_model_img = np.stack(aosTable_matched['zk_intrinsic'])[img_mask]
    fit_col = f'zk_fit_{fit_prefix}'
    if fit_col in aosTable_matched.colnames:
        zk_fit_img = np.stack(aosTable_matched[fit_col])[img_mask]
    else:
        zk_fit_img = np.stack(aosTable_matched['zk_fit'])[img_mask]

    bins_edge = np.linspace(-fpradius, fpradius, n_steps)
    n_rows, n_cols = 3, 4

    fig = plt.figure(figsize=(18, 12))
    width_ratios = []
    for c in range(n_cols):
        width_ratios.extend([20, 1])
    gs = GridSpec(n_rows, n_cols * 2, figure=fig, width_ratios=width_ratios,
                  wspace=0.4, hspace=0.35)

    band_str = f'  band={band}' if band else ''
    fig.suptitle(f'Single-Image Residuals: day_obs={day_obs}  seq_num={seq_num}{band_str}'
                 f'  ({n_donuts} donuts)', fontsize=14, y=0.98)

    for ax_idx, iZ in enumerate(iZs_plot):
        row = ax_idx // n_cols
        col = ax_idx % n_cols
        ax = fig.add_subplot(gs[row, col * 2])
        cax = fig.add_subplot(gs[row, col * 2 + 1])
        j = iZidx[iZ]
        resid = zk_data_img[:, j] - zk_model_img[:, j] * 1e6 - zk_fit_img[:, j]

        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, resid, statistic=statistic, bins=[bins_edge, bins_edge])
        finite_vals = stat_val[np.isfinite(stat_val)]
        vmin, vmax = (np.percentile(finite_vals, [plo, phi]) if len(finite_vals) > 0
                      else (-1.0, 1.0))

        im = ax.imshow(stat_val.T, origin='lower',
                       extent=[-fpradius, fpradius, -fpradius, fpradius],
                       cmap='RdBu_r', interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        fig.colorbar(im, cax=cax)
        ax.set_title(f'Z{iZ}', fontsize=11, loc='center')
        ax.set_xlim(-fpradius, fpradius)
        ax.set_ylim(-fpradius, fpradius)
        if col == 0:
            ax.set_ylabel(f'thx_{coord_sys} [deg]')
        if row == n_rows - 1:
            ax.set_xlabel(f'thy_{coord_sys} [deg]')

        # c1 (piston) bar indicator
        if iZ in c1_vals:
            c1_val = np.clip(c1_vals[iZ], -k1_range, k1_range)
            bar_anchor, bar_y, bar_h = 0.25, 1.04, 0.03
            bar_dx = c1_val / k1_range * 0.25
            color = 'steelblue' if c1_val >= 0 else 'firebrick'
            rect_x = min(bar_anchor, bar_anchor + bar_dx)
            rect_w = abs(bar_dx)
            if rect_w > 0.001:
                rect = Rectangle((rect_x, bar_y - bar_h / 2), rect_w, bar_h,
                                 transform=ax.transAxes, clip_on=False,
                                 facecolor=color, edgecolor='none', alpha=0.7)
                ax.add_patch(rect)
            ax.plot([bar_anchor, bar_anchor], [bar_y - bar_h / 2, bar_y + bar_h / 2],
                    transform=ax.transAxes, color='black', linewidth=0.8, clip_on=False)
            ax.text(bar_anchor + bar_dx, bar_y + bar_h / 2 + 0.01,
                    f'{c1_vals[iZ]:.2f}', transform=ax.transAxes,
                    fontsize=7, ha='center', va='bottom', color=color)

    output_file = f'{output_dir}/single_image_resid_{day_obs}_{seq_num}.jpg'
    fig.savefig(output_file, dpi=120, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return output_file


def plot_zernike_trio(aosTable_matched, iZ, plo=4.0, phi=96.0,
                      statistic='median', fit_prefix='z13',
                      output_dir='.', date_range_str='', pdf=None):
    """Create trio of plots for a Zernike term: Data, Model, Data-Model.
    
    Parameters
    ----------
    statistic : str
        Statistic for binned_statistic_2d (default 'median').
    pdf : PdfPages or None
        If provided, save figure to this PDF instead of a standalone PNG.
    """
    nsteps = 18 * 4 + 1
    fpradius = 1.8
    xbins = np.linspace(-fpradius, fpradius, nsteps)
    ybins = np.linspace(-fpradius, fpradius, nsteps)
    
    xval = np.rad2deg(aosTable_matched[f'thy_{coord_sys}_extra'])
    yval = np.rad2deg(aosTable_matched[f'thx_{coord_sys}_extra'])
    
    zk_data_all = np.stack(aosTable_matched[f'zk_{coord_sys}'])
    fit_col = f'zk_fit_{fit_prefix}'
    if fit_col in aosTable_matched.colnames:
        zk_fit_all = np.stack(aosTable_matched[fit_col])
    else:
        zk_fit_all = np.stack(aosTable_matched['zk_fit'])
    zk_model_all = np.stack(aosTable_matched['zk_intrinsic'])
    
    zval_data = zk_data_all[:, iZidx[iZ]] - zk_fit_all[:, iZidx[iZ]]
    zval_model = zk_model_all[:, iZidx[iZ]] * 1e6
    zval_residual = zval_data - zval_model
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    for ax, zval, cmap, title_str in [
        (axes[0], zval_data, 'viridis', f'Z{iZ} Data (linear fit subtracted)'),
        (axes[1], zval_model, 'viridis', f'Z{iZ} Model Intrinsic'),
        (axes[2], zval_residual, 'RdBu_r', f'Z{iZ} Data - Model'),
    ]:
        stat_val, _, _, _ = binned_statistic_2d(
            xval, yval, zval, statistic=statistic, bins=[xbins, ybins])
        vmin, vmax = np.nanpercentile(zval, [plo, phi])
        im = ax.imshow(stat_val.T, origin='lower',
                       extent=[xbins[0], xbins[-1], ybins[0], ybins[-1]],
                       cmap=cmap, interpolation='none', aspect='equal',
                       vmin=vmin, vmax=vmax)
        add_colorbar(im, label='μm')
        ax.set_xlabel(f'thy_{coord_sys} [deg]')
        ax.set_ylabel(f'thx_{coord_sys} [deg]')
        ax.set_title(title_str)
        ax.set_aspect('equal')
    
    title = f'Z{iZ} Comparison ({statistic})'
    if date_range_str:
        title += f' ({date_range_str})'
    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    
    if pdf is not None:
        pdf.savefig(fig, dpi=150, bbox_inches='tight')
    else:
        output_file = f'{output_dir}/Z{iZ}_trio_comparison.png'
        fig.savefig(output_file, dpi=150, bbox_inches='tight')
        print(f"Saved: {output_file}")
    
    plt.show()
    
    print(f"Z{iZ}: Data σ={np.nanstd(zval_data):.2f}, Model σ={np.nanstd(zval_model):.2f}, "
          f"Resid σ={np.nanstd(zval_residual):.2f} μm")


print("Helper functions loaded:")
print("  focal_plane_zernike_basis(), fit_focal_zernikes()")
print("  plot_fit_params_and_residuals(), plot_single_image_residual_grid(), plot_zernike_trio()")
print(f"\nField angle columns: thx_{coord_sys}, thy_{coord_sys}")
print(f"Zernike data column: zk_{coord_sys}")

In [ ]:
# Extract Zernike arrays and determine Noll indices from visit_info
zk_data = np.stack(aosTable[f'zk_{coord_sys}'])
zk_model = np.stack(aosTable['zk_intrinsic'])
nZk = zk_data.shape[1]

# Read Noll indices from visit_info (should be uniform across visits)
if visit_info is not None and 'nollIndices' in visit_info.colnames:
    # nollIndices is an array column — take the first row (all should be identical)
    noll_arr = np.array(visit_info['nollIndices'][0])
    iZs = [int(n) for n in noll_arr]
    if len(iZs) != nZk:
        print(f"WARNING: nollIndices length ({len(iZs)}) != zk array width ({nZk})")
        print(f"  nollIndices: {iZs}")
        print(f"  Falling back to contiguous range")
        iZs = list(range(4, 4 + nZk))
    else:
        print(f"Noll indices from visit_info: {iZs}")
else:
    # Fallback: assume contiguous starting at 4
    if nZk == 19:
        iZs = list(range(4, 23))
    else:
        iZs = list(range(4, 4 + nZk))
    print(f"Warning: nollIndices not in visit_info — assuming contiguous Z4-Z{iZs[-1]}")

iZidx = {iZ: i for i, iZ in enumerate(iZs)}

print(f"\nZernike arrays: shape {zk_data.shape} (rows x Zernikes)")
print(f"  Noll indices ({len(iZs)} terms): {iZs}")
print(f"  Coordinate system: {coord_sys}")

# Determine which Zernikes to use for 3x4 subplot grids (first 12 available)
iZs_plot_12 = iZs[:12] if len(iZs) >= 12 else iZs
print(f"  Default 3x4 plot set: {iZs_plot_12}")

<a id='analysis'></a>
## Analysis

Filter for matched intra/extra donuts, then fit per-image linear model and create trio plots.

In [8]:
# Filter for matched intra/extra donuts
matched_mask = aosTable['matched_intra_extra']
print(f"Total donuts: {len(aosTable)}")
print(f"Matched intra/extra donuts: {np.sum(matched_mask)}")
print(f"Fraction matched: {np.sum(matched_mask)/len(aosTable):.3f}")

# Apply filter
aosTable_matched = aosTable[matched_mask]

Total donuts: 448063
Matched intra/extra donuts: 323608
Fraction matched: 0.722


<a id='fit'></a>
## Focal-Plane Zernike Fits per Image

Two fits per image and Zernike term:
- **z13**: focal Noll Z1–Z3 (piston + tilt + tip) — linear model
- **z16**: focal Noll Z1–Z6 (adds defocus + astigmatism) — quadratic model

Both use robust regression (Huber's T) and subtract the intrinsic model.

In [ ]:
# Fit 1: focal Noll Z1-Z3 (piston + tilt + tip)
fit_table_z13, zk_fit_z13, zk_wgt_z13 = fit_focal_zernikes(
    aosTable_matched, iZs, iZidx, coord_sys,
    max_focal_noll=3, include_intrinsic=True, prefix='z13')

aosTable_matched['zk_fit_z13'] = list(zk_fit_z13)
aosTable_matched['zk_rlm_weights_z13'] = list(zk_wgt_z13)

# Also keep 'zk_fit' as alias for the primary (z13) fit for backward compat
aosTable_matched['zk_fit'] = list(zk_fit_z13)

print(f"\nSample z13 (Z4): c1 range = [{fit_table_z13['z13_z4_c1'].min():.3f}, "
      f"{fit_table_z13['z13_z4_c1'].max():.3f}] μm")

# Fit 2: focal Noll Z1-Z6 (adds defocus + astigmatism)
fit_table_z16, zk_fit_z16, zk_wgt_z16 = fit_focal_zernikes(
    aosTable_matched, iZs, iZidx, coord_sys,
    max_focal_noll=6, include_intrinsic=True, prefix='z16')

aosTable_matched['zk_fit_z16'] = list(zk_fit_z16)
aosTable_matched['zk_rlm_weights_z16'] = list(zk_wgt_z16)

print(f"\nSample z16 (Z4): c1 range = [{fit_table_z16['z16_z4_c1'].min():.3f}, "
      f"{fit_table_z16['z16_z4_c1'].max():.3f}] μm")
print(f"  c4 (defocus) range = [{fit_table_z16['z16_z4_c4'].min():.3f}, "
      f"{fit_table_z16['z16_z4_c4'].max():.3f}] μm")

In [ ]:
# Merge z13 and z16 fit tables, then merge with visit_info, and save

# Combine z13 and z16 fits (join on day_obs, seq_num)
# z13 has the shared columns (day_obs, seq_num, image_idx, n_donuts)
# z16 has its own coefficient columns — drop the duplicate shared columns
z16_cols_only = [c for c in fit_table_z16.colnames if c.startswith('z16_')]
fit_table_combined = fit_table_z13.copy()
for col in z16_cols_only:
    fit_table_combined[col] = fit_table_z16[col]

print(f"Combined fit table: {len(fit_table_combined)} rows, {len(fit_table_combined.columns)} columns")

# Merge with visit_info (if available)
if visit_info is not None:
    # Join on day_obs + seq_num
    from astropy.table import join
    # Ensure matching column types
    fit_merged = join(fit_table_combined, visit_info, keys=['day_obs', 'seq_num'],
                      join_type='left')
    print(f"Merged with visit_info: {len(fit_merged)} rows, {len(fit_merged.columns)} columns")
    print(f"  Visit_info columns added: {[c for c in visit_info.colnames if c not in ['day_obs', 'seq_num']]}")
else:
    fit_merged = fit_table_combined
    print("visit_info not available — skipping merge")

# Save as parquet
output_fits_file = (f'{output_dir}/intrinsic_fits_{wep_ver}_{dviz_ver}'
                    f'_{day_obs_min}_{day_obs_max}.parquet')
fit_merged.write(output_fits_file, format='parquet', overwrite=True)
print(f"\nSaved: {output_fits_file}")
print(f"  {len(fit_merged)} rows x {len(fit_merged.columns)} columns")

# Use fit_table_z13 as the primary fit_table for downstream plots
fit_table = fit_table_z13

<a id='fitplots'></a>
## Fit Parameter Plots and Residual Histograms

Loop over each day_obs: show the first day inline, save all to output/.

In [ ]:
# Get list of day_obs values remaining after filtering
all_day_obs = sorted(set(np.array(aosTable_matched['day_obs']).tolist()))
print(f"Day_obs values for plots: {all_day_obs}")

# Build arrays needed for masking
day_obs_matched = np.array(aosTable_matched['day_obs'])
fit_day_obs = np.array(fit_table['day_obs'])

In [ ]:
# Plot fit parameters and residual histograms for each day_obs
# First day shown inline; all days saved to output/ as PDFs
# Make plots for both z13 and z16 fits

fit_day_obs_z16 = np.array(fit_table_z16['day_obs'])

for fit_prefix, ft in [('z13', fit_table), ('z16', fit_table_z16)]:
    ft_day_obs = np.array(ft['day_obs'])
    
    for i, dobs in enumerate(all_day_obs):
        day_mask_fit = ft_day_obs == dobs
        day_mask_matched = day_obs_matched == dobs
        ft_day = ft[day_mask_fit]
        
        show_inline = (i == 0 and fit_prefix == 'z13')  # only show first z13 day inline
        plot_fit_params_and_residuals(
            ft_day, aosTable_matched, day_mask_matched,
            day_obs_list=[dobs], fit_prefix=fit_prefix,
            iZs_fit_plot=iZs_plot_12, iZs_hist=iZs_plot_12,
            output_dir=output_dir, show=show_inline)
    
    # Combined all-days plots
    plot_fit_params_and_residuals(
        ft, aosTable_matched, np.ones(len(aosTable_matched), dtype=bool),
        day_obs_list=all_day_obs, fit_prefix=fit_prefix,
        iZs_fit_plot=iZs_plot_12, iZs_hist=iZs_plot_12,
        output_dir=output_dir, show=(fit_prefix == 'z13'))

<a id='singleimage'></a>
## Single-Image Residual Maps

For each visit, plot a 4x3 grid of binned residual maps (Z4–Z15) using `binned_statistic_2d`.
Show the first visit inline; all visits are saved as JPEGs and combined into an MPEG movie.

In [ ]:
# Generate single-image residual maps for all visits
# First visit shown inline, rest saved to output/ for movie

# Build a lookup for band from visit_info (if available)
band_lookup = {}
if visit_info is not None:
    for row in visit_info:
        band_lookup[(int(row['day_obs']), int(row['seq_num']))] = str(row['band'])

# Get sorted list of all (day_obs, seq_num) in matched table
all_images = sorted(set(zip(
    np.array(aosTable_matched['day_obs']).tolist(),
    np.array(aosTable_matched['seq_num']).tolist()
)))
print(f"Generating residual maps for {len(all_images)} visits...")
print(f"  Zernike subplot set: {iZs_plot_12}")

frame_files = []
for i, (dobs, snum) in enumerate(all_images):
    band = band_lookup.get((dobs, snum), '')
    show_inline = (i == 0)
    outfile = plot_single_image_residual_grid(
        aosTable_matched, dobs, snum,
        band=band, fit_table=fit_table, fit_prefix='z13',
        iZs_plot=iZs_plot_12,
        n_steps=16, statistic='median',
        plo=2.0, phi=98.0,
        output_dir=output_dir, show=show_inline
    )
    if outfile is not None:
        frame_files.append(outfile)
    if (i + 1) % 50 == 0:
        print(f"  ...processed {i + 1}/{len(all_images)} visits")

print(f"Generated {len(frame_files)} residual map frames")

# Combine into MPEG movie using ffmpeg
movie_success = False
if len(frame_files) > 1:
    list_file = f'{output_dir}/frame_list.txt'
    with open(list_file, 'w') as f:
        for fpath in frame_files:
            f.write(f"file '{Path(fpath).name}'\n")
            f.write("duration 0.5\n")
    
    movie_file = f'{output_dir}/single_image_residuals.mp4'
    try:
        result = subprocess.run(
            ['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
             '-i', 'frame_list.txt',
             '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
             '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
             '-r', '2',
             'single_image_residuals.mp4'],
            capture_output=True, text=True, cwd=output_dir
        )
        if result.returncode == 0:
            print(f"Saved movie: {movie_file}")
            movie_success = True
        else:
            print(f"ffmpeg failed (return code {result.returncode}):")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    except FileNotFoundError:
        print("ffmpeg not found — movie not created. Install ffmpeg to generate MPEG.")

# Clean up individual JPEGs and frame_list.txt if movie was successfully created
if movie_success:
    for fpath in frame_files:
        Path(fpath).unlink(missing_ok=True)
    Path(list_file).unlink(missing_ok=True)
    print(f"Cleaned up {len(frame_files)} individual JPEGs and frame_list.txt")
else:
    print("Keeping individual JPEGs (movie was not created).")

<a id='viz'></a>
## Visualizations

Create trio plots for each Zernike term: Data (linear fit subtracted), Model, and Data-Model residuals.

In [ ]:
# Build day_obs label for trio plot titles
if len(all_day_obs) <= 4:
    day_obs_plot_str = 'day_obs: ' + ', '.join(str(d) for d in all_day_obs)
else:
    day_obs_plot_str = f'day_obs: {all_day_obs[0]}...{all_day_obs[-1]} ({len(all_day_obs)} days)'

aosTable_plot = aosTable_matched
print(f"Donuts for trio plots: {len(aosTable_plot)} ({day_obs_plot_str})")

# All trio plots into a single PDF
trio_pdf_path = f'{output_dir}/trio_comparison_all.pdf'
with PdfPages(trio_pdf_path) as trio_pdf:
    for iZ in iZs:
        plot_zernike_trio(aosTable_plot, iZ=iZ, plo=plo_default, phi=phi_default,
                          statistic='median', fit_prefix='z13',
                          output_dir=output_dir, date_range_str=day_obs_plot_str,
                          pdf=trio_pdf)

print(f"\nSaved all trio plots to: {trio_pdf_path}")